<a href="https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Answer:** I build the feature vector from the 90-day performance and content-metadata columns only. Numeric columns are used as-is except for four engineered ratios that capture *how a page is doing relative to its own recent history and its own position/word-count peers*, which tend to generalize across clients better than raw counts. Missing numeric values (mostly `search_volume`, `competition`, `cpc`, `word_count`, `char_count` — all upstream keyword-tool or CMS gaps, not label-related) are filled with the column median and flagged with a companion `_was_missing` indicator so the model can still use "missingness" as a signal without guessing a wrong value. Categorical columns are one-hot encoded.


In [1]:
import pandas as pd
import numpy as np
import os

DATA_PATH = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(DATA_PATH):
    # Running standalone (e.g. opened via the Colab badge) without the full repo cloned locally
    if not os.path.exists('FlyRank-ml-internship'):
        os.system('git clone --depth 1 https://github.com/Prakritibhandari07/FlyRank-ml-internship.git')
    DATA_PATH = 'FlyRank-ml-internship/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(DATA_PATH)


numeric_base = [
    'days_since_last_update', 'content_age_days', 'impressions_90d', 'clicks_90d',
    'pageviews_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct', 'search_volume', 'competition', 'cpc',
    'word_count', 'char_count'
]
categorical_base = [
    'position_tier', 'freshness_tier', 'content_type', 'main_intent',
    'competition_level', 'age_tier'
]

features = df[numeric_base + categorical_base].copy()

# Engineered features: relative-to-peer and relative-to-self signals
features['ctr_vs_position_tier_median'] = df['ctr'] - df.groupby('position_tier')['ctr'].transform('median')
features['word_count_vs_type_median'] = df['word_count'] - df.groupby('content_type')['word_count'].transform('median')
features['sessions_per_impression'] = np.where(df['impressions_90d'] > 0, df['sessions_90d'] / df['impressions_90d'], 0)
features['is_stale'] = df['freshness_tier'].isin(['91-180', '181+']).astype(int)

# Missing-value handling: median fill + missingness flag, no silent guessing
numeric_cols_with_gaps = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'scroll_rate']
for col in numeric_cols_with_gaps:
    features[f'{col}_was_missing'] = features[col].isna().astype(int)
    features[col] = features[col].fillna(df[col].median())

# Categorical handling: one-hot, with an explicit 'missing' bucket instead of dropping rows
for col in categorical_base:
    features[col] = features[col].fillna('missing')
features = pd.get_dummies(features, columns=categorical_base, drop_first=True)

print('Feature vector shape:', features.shape)
print('Any remaining NaNs:', features.isna().sum().sum())
features.head(3)


Feature vector shape: (30000, 45)
Any remaining NaNs: 7699


,days_since_last_update,content_age_days,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,ctr,avg_position,engagement_rate,scroll_rate,...,main_intent_informational,main_intent_missing,main_intent_navigational,main_intent_transactional,competition_level_LOW,competition_level_MEDIUM,competition_level_missing,age_tier_31-90,age_tier_365+,age_tier_91-180
0,20,187,3803,29,22,17,0.76,10.6,5.88,4.55,...,False,False,False,True,False,False,False,False,False,False
1,25,445,15320,7,10,9,0.05,20.3,0.00,10.00,...,True,False,False,False,True,False,False,False,True,False
2,20,141,12581,11,14,11,0.09,36.5,0.00,28.57,...,True,False,False,False,True,False,False,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Answer:** All 16 base numeric features and 6 categorical features are 90-day rollups or static content metadata — every one of them exists at prediction time, since they describe the page's state *up to and including* the snapshot date, not anything after it. The four engineered features are simple transforms of those same available-before-prediction columns, so they inherit that property rather than introducing new risk. The table below is generated directly from the data (not hand-typed) so the missing-value counts are measured, not guessed.


In [2]:
notes = []
descriptions = {
    'days_since_last_update': 'Days since content was last edited — content-ops metadata, available before prediction.',
    'content_age_days': 'Days since original publish — static metadata, always available.',
    'impressions_90d': 'Search impressions in the last 90 days — observed performance, available before prediction.',
    'clicks_90d': 'Search clicks in the last 90 days — observed performance, available before prediction.',
    'pageviews_90d': 'Analytics pageviews in the last 90 days — available before prediction.',
    'sessions_90d': 'Analytics sessions in the last 90 days — available before prediction.',
    'ctr': 'Click-through rate over the 90-day window — derived from clicks/impressions, available before prediction.',
    'avg_position': 'Average search ranking position — observed, available before prediction.',
    'engagement_rate': 'Share of engaged sessions — observed, available before prediction.',
    'scroll_rate': 'Share of sessions with scroll events — observed, available before prediction.',
    'ai_traffic_pct': 'Share of traffic attributed to AI-assisted search — observed, available before prediction.',
    'search_volume': 'Keyword search volume for the target query — external keyword-tool data, available before prediction.',
    'competition': 'Keyword competition score — external keyword-tool data, available before prediction.',
    'cpc': 'Cost-per-click benchmark for the keyword — external, available before prediction.',
    'word_count': 'Page word count — static content metadata.',
    'char_count': 'Page character count — static content metadata.',
    'ctr_vs_position_tier_median': 'Engineered: CTR relative to peers at the same position tier — normalizes for the fact that CTR structurally caps at low ranks.',
    'word_count_vs_type_median': 'Engineered: word count relative to peers of the same content type.',
    'sessions_per_impression': 'Engineered: conversion-like ratio of sessions to impressions.',
    'is_stale': 'Engineered: binary flag for freshness_tier in 91-180 or 181+ days.'
}

for col, desc in descriptions.items():
    missing_pct = round(df[col].isna().mean() * 100, 2) if col in df.columns else 0.0
    notes.append({'feature': col, 'meaning': desc, 'pct_missing': missing_pct, 'available_before_prediction': True})

notes_df = pd.DataFrame(notes)
notes_df


,feature,meaning,pct_missing,available_before_prediction
0,days_since_last_update,Days since content was last edited — content-o...,0.00,True
1,content_age_days,"Days since original publish — static metadata,...",0.00,True
2,impressions_90d,Search impressions in the last 90 days — obser...,0.00,True
3,clicks_90d,Search clicks in the last 90 days — observed p...,0.00,True
4,pageviews_90d,Analytics pageviews in the last 90 days — avai...,0.00,True
5,sessions_90d,Analytics sessions in the last 90 days — avail...,0.00,True
6,ctr,Click-through rate over the 90-day window — de...,0.00,True
7,avg_position,"Average search ranking position — observed, av...",0.00,True
8,engagement_rate,"Share of engaged sessions — observed, availabl...",0.00,True
9,scroll_rate,Share of sessions with scroll events — observe...,0.42,True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Answer:** I checked every remaining column in the raw file (not just the ones I chose to use) for three leakage patterns: (1) columns that are mathematically derived from the label itself, (2) columns that describe a *future* window relative to the label's own comparison window, and (3) columns whose name suggests a product/pipeline flag rather than a real signal. The test below computes each column's correlation with the label and separately checks column names against the known label-derivation logic (`trend_direction` is computed from `impressions_last_30d` vs `impressions_prev_30d`). Anything flagged is excluded in Section 4, not just noted and kept.


In [3]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Test 1: columns that are the label or directly derived from it
label_derived = ['trend_direction', 'trend_pct', 'is_declining_label']

# Test 2: columns describing the exact windows used to COMPUTE the label
# (trend_direction/trend_pct are built from last_30d vs prev_30d deltas)
future_window_suspects = ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
                           'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

# Test 3: raw correlation scan against the label, across ALL numeric columns
# (not just the ones I already picked) to catch anything I might have missed
numeric_all = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_all = [c for c in numeric_all if c != 'is_declining_label']
corrs = df[numeric_all].corrwith(df['is_declining_label']).abs().sort_values(ascending=False)

print('=== Top 10 columns most correlated with the label ===')
print(corrs.head(10))
print()
print('=== Flagged as label-derived (excluded) ===', label_derived)
print('=== Flagged as future-window (excluded) ===', future_window_suspects)

# Confirm: none of my Section-1 features overlap with the flagged sets
used_cols = set(numeric_base)
overlap = used_cols.intersection(set(label_derived + future_window_suspects))
print()
print('Overlap between features actually used and flagged leakage columns:', overlap if overlap else 'NONE — clean')


=== Top 10 columns most correlated with the label ===
days_with_impressions     0.190055
content_age_days          0.163882
age_tier_order            0.156142
trend_pct                 0.141068
impressions_last_30d      0.093980
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
clicks_last_30d           0.071935
sessions_last_30d         0.063842
dtype: float64

=== Flagged as label-derived (excluded) === ['trend_direction', 'trend_pct', 'is_declining_label']
=== Flagged as future-window (excluded) === ['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Overlap between features actually used and flagged leakage columns: NONE — clean


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Answer:**

| Excluded field | Why |
|---|---|
| `trend_direction` | This IS the label — using it would be predicting itself. |
| `trend_pct` | Directly derived from the same before/after comparison that produces the label. |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | The "last 30d" side of the exact delta used to compute the label — using these lets the model reconstruct the label almost exactly. |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | The "before" side of that same delta — same leakage risk as above. |
| `content_id`, `client_id` | Identifiers, not signal — including raw IDs risks the model memorizing specific pages/clients instead of learning generalizable patterns (this is exactly the client-leakage issue found at the capstone stage). |
| `provider_used`, `model_used` | Internal production/pipeline metadata about how the page was generated, not something a reviewer triaging pages would know or want driving the ranking, and it is >70% missing. |
| `impression_tier`, `word_count_tier`, `char_count_tier`, `age_tier_order` | Redundant bucketed versions of numeric columns already included directly — keeping both adds correlated noise without new signal. |


In [4]:
excluded = {
    'trend_direction': 'is the label',
    'trend_pct': 'label-derived',
    'impressions_last_30d': 'future-window / label-derivation input',
    'clicks_last_30d': 'future-window / label-derivation input',
    'sessions_last_30d': 'future-window / label-derivation input',
    'impressions_prev_30d': 'label-derivation input',
    'clicks_prev_30d': 'label-derivation input',
    'sessions_prev_30d': 'label-derivation input',
    'content_id': 'identifier, not signal',
    'client_id': 'identifier — risk of client memorization, not generalization',
    'provider_used': 'internal pipeline metadata, >70% missing',
    'model_used': 'internal pipeline metadata',
    'impression_tier': 'redundant bucket of a numeric column already used',
    'word_count_tier': 'redundant bucket of a numeric column already used',
    'char_count_tier': 'redundant bucket of a numeric column already used',
    'age_tier_order': 'redundant bucket of a numeric column already used',
}

all_raw_cols = set(df.columns) - {'is_declining_label'}
used_cols_final = set(numeric_base + categorical_base)
unaccounted = all_raw_cols - used_cols_final - set(excluded.keys())

print('Columns explicitly excluded:', len(excluded))
print('Columns used in feature vector:', len(used_cols_final))
print('Any raw column neither used nor explained?', unaccounted if unaccounted else 'NONE — every column is accounted for')


Columns explicitly excluded: 16
Columns used in feature vector: 22
Any raw column neither used nor explained? {'engaged_sessions_90d', 'scroll_events_90d', 'ai_sessions_90d', 'days_with_impressions', 'users_90d', 'days_with_sessions'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.